# NCAA Archetype Classification

This notebook classifies players into the three management archetypes without modifying any source datasets.

Buckets:
- PG / Combo Guard
- 2-4 Interchangeable Wing
- 5 / Stretch 4 / Big Wing

The notebook uses management's preferred parameters as soft scoring rules. Players who miss a parameter are not removed; they receive lower scores and fit-tier notes.

## Data Sources

- D-I: `mbb_with_pca.csv`
- D-II: `d2_data_cleaned.csv`
- D-III: `d3_data_cleaned.csv`

`d2_mens_database.csv` is not used here because it is rawer and lacks derived fields such as `three_share`, `AST_TOV`, and per-40 stats. `data.csv` appears to be a smaller/older processed D-II subset, so the fuller `d2_data_cleaned.csv` is used instead.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

HERE = Path.cwd()

D1_PATH = HERE / "mbb_with_pca.csv"
D2_PATH = HERE / "d2_data_cleaned.csv"
D3_PATH = HERE / "d3_data_cleaned.csv"

D1_PATH.exists(), D2_PATH.exists(), D3_PATH.exists()

## Parameters From Management

These are encoded as preferred thresholds, not hard exclusions.

- All players baseline: high assist creation, `eFG >= .500`, `3P% >= .300`, high A/TO, strong defensive rebounding.
- PG / Combo Guard: high assist creation, `3P% >= .330`, `3P rate >= .300`, positive A/TO.
- 2-4 Wing: strong defensive rebounding, `3P% >= .330`, `3P rate >= .300`, positive A/TO.
- 5 / Stretch 4 / Big Wing: `height >= 6'7\"`, strong defensive rebounding, `3P% >= .300`, `3P rate >= .250`, positive A/TO.

D-I has true `AST_pct` and `DRB_pct`. D-II/D-III do not, so this notebook uses `ast_per_40` and `DRBPG` as proxies for those components.

In [ ]:
THRESHOLDS = {
    "baseline_efg": 0.500,
    "baseline_3p_pct": 0.300,
    "pg_3p_pct": 0.330,
    "pg_3p_rate": 0.300,
    "wing_3p_pct": 0.330,
    "wing_3p_rate": 0.300,
    "big_3p_pct": 0.300,
    "big_3p_rate": 0.250,
    "big_min_height_in": 79,
    "positive_ast_tov": 1.000,
    "high_percentile": 70,
    "d1_dreb_rate": 15.0,
    "proxy_dreb_percentile": 60,
}

ARCHETYPE_LABELS = {
    "score_pg_combo": "PG / Combo Guard",
    "score_wing_2_4": "2-4 Interchangeable Wing",
    "score_stretch_big": "5 / Stretch 4 / Big Wing",
}

## Load And Normalize Datasets

The normalized table keeps the source values plus common archetype fields. No source CSV is changed.

In [ ]:
D1_ROLE_MAP = {
    "Pure PG": "G",
    "Scoring PG": "G",
    "Combo G": "G",
    "Wing G": "G/F",
    "Wing F": "F",
    "Stretch 4": "F/C",
    "PF/C": "F/C",
    "C": "C",
}

def flip_name(value):
    value = "" if pd.isna(value) else str(value).strip()
    if "," not in value:
        return value
    last, first = value.split(",", 1)
    return f"{first.strip()} {last.strip()}".strip()

def normalize_position(value):
    value = "" if pd.isna(value) else str(value).strip().upper()
    if value in {"G", "G/F", "F", "F/C", "C"}:
        return value
    if value.startswith("GUARD") or value == "PG":
        return "G"
    if value.startswith("FORWARD"):
        return "F"
    if value.startswith("CENTER"):
        return "C"
    if value.startswith("G/F") or value.startswith("GF"):
        return "G/F"
    if value.startswith("F/C") or value.startswith("FC"):
        return "F/C"
    if value.startswith("G"):
        return "G"
    if value.startswith("F"):
        return "F"
    if value.startswith("C"):
        return "C"
    return "G"

def numeric(series, default=0):
    return pd.to_numeric(series, errors="coerce").fillna(default)

def load_d1(path):
    raw = pd.read_csv(path)
    out = pd.DataFrame({
        "division": "D-I",
        "source_file": path.name,
        "name": raw["player_name"].astype(str).str.strip(),
        "team": raw["team"].astype(str).str.strip(),
        "conference": raw["conf"].astype(str).str.strip(),
        "pos": raw["role"].map(D1_ROLE_MAP).fillna("G"),
        "role": raw["role"].astype(str),
        "height_in": numeric(raw["height_inches"], 78),
        "gp": numeric(raw["GP"]),
        "mpg": numeric(raw["mins_per_game"]),
        "ppg": numeric(raw["pts_per_game"]),
        "apg": numeric(raw["ast_per_game"]),
        "rpg": numeric(raw["treb_per_game"]),
        "three_pct": numeric(raw["3P_pct"]),
        "three_rate": numeric(raw["three_share"]),
        "efg": numeric(raw["eFG"]) / 100.0,
        "ast_tov": numeric(raw["AST_TOV"]),
        "assist_creation": numeric(raw["AST_pct"]),
        "assist_source": "AST_pct",
        "dreb": numeric(raw["DRB_pct"]),
        "dreb_source": "DRB_pct",
        "has_true_dreb_rate": True,
    })
    return out

def load_cleaned_division(path, division):
    raw = pd.read_csv(path)
    out = pd.DataFrame({
        "division": division,
        "source_file": path.name,
        "name": raw["Player Name"].apply(flip_name),
        "team": raw["Team"].astype(str).str.strip(),
        "conference": raw["Conference"].astype(str).str.strip(),
        "pos": raw["Position"].apply(normalize_position),
        "role": raw["Position"].astype(str),
        "height_in": numeric(raw["Height"], 72),
        "gp": numeric(raw["GP"]),
        "mpg": numeric(raw["MPG"]),
        "ppg": numeric(raw["PPG"]),
        "apg": numeric(raw["APG"]),
        "rpg": numeric(raw["RPG"]),
        "three_pct": numeric(raw["3PT%"]),
        "three_rate": numeric(raw["three_share"]),
        "efg": numeric(raw["eFG"]),
        "ast_tov": numeric(raw["AST_TOV"]),
        "assist_creation": numeric(raw["ast_per_40"]),
        "assist_source": "ast_per_40 proxy",
        "dreb": numeric(raw["DRBPG"]),
        "dreb_source": "DRBPG proxy",
        "has_true_dreb_rate": False,
    })
    return out

players = pd.concat([
    load_d1(D1_PATH),
    load_cleaned_division(D2_PATH, "D-II"),
    load_cleaned_division(D3_PATH, "D-III"),
], ignore_index=True)

players["player_key"] = players["division"] + " | " + players["team"] + " | " + players["name"]
players.shape

In [ ]:
players.head()

## Build Percentile Features

Percentiles are calculated within division. Defensive rebounding also gets a position-adjusted percentile because management said the expectation can fluctuate by position.

In [ ]:
def percentile_by(group_cols, value_col):
    return players.groupby(group_cols)[value_col].rank(pct=True, method="average") * 100

players["pct_assist_creation"] = percentile_by("division", "assist_creation")
players["pct_three_pct"] = percentile_by("division", "three_pct")
players["pct_three_rate"] = percentile_by("division", "three_rate")
players["pct_ast_tov"] = percentile_by("division", "ast_tov")
players["pct_efg"] = percentile_by("division", "efg")
players["pct_dreb"] = percentile_by("division", "dreb")
players["pct_dreb_pos_adj"] = percentile_by(["division", "pos"], "dreb")
players["pct_size"] = percentile_by("division", "height_in")

players[["division", "name", "pos", "assist_creation", "assist_source", "dreb", "dreb_source", "pct_assist_creation", "pct_dreb_pos_adj"]].head()

## Threshold Flags

For D-I, the defensive rebounding baseline uses `DRB_pct >= 15`. For D-II/D-III, this uses a proxy: position-adjusted defensive rebounding percentile >= 60.

In [ ]:
players["meets_high_assist"] = players["pct_assist_creation"] >= THRESHOLDS["high_percentile"]
players["meets_efg_500"] = players["efg"] >= THRESHOLDS["baseline_efg"]
players["meets_3p_300"] = players["three_pct"] >= THRESHOLDS["baseline_3p_pct"]
players["meets_high_ast_tov"] = players["pct_ast_tov"] >= THRESHOLDS["high_percentile"]
players["meets_positive_ast_tov"] = players["ast_tov"] > THRESHOLDS["positive_ast_tov"]

players["meets_dreb_profile"] = np.where(
    players["has_true_dreb_rate"],
    players["dreb"] >= THRESHOLDS["d1_dreb_rate"],
    players["pct_dreb_pos_adj"] >= THRESHOLDS["proxy_dreb_percentile"],
)

players["meets_all_player_baseline"] = (
    players["meets_high_assist"]
    & players["meets_efg_500"]
    & players["meets_3p_300"]
    & players["meets_high_ast_tov"]
    & players["meets_dreb_profile"]
)

players["meets_pg_preferred"] = (
    players["meets_high_assist"]
    & (players["three_pct"] >= THRESHOLDS["pg_3p_pct"])
    & (players["three_rate"] >= THRESHOLDS["pg_3p_rate"])
    & players["meets_positive_ast_tov"]
)

players["meets_wing_preferred"] = (
    players["meets_dreb_profile"]
    & (players["three_pct"] >= THRESHOLDS["wing_3p_pct"])
    & (players["three_rate"] >= THRESHOLDS["wing_3p_rate"])
    & players["meets_positive_ast_tov"]
)

players["meets_big_preferred"] = (
    (players["height_in"] >= THRESHOLDS["big_min_height_in"])
    & players["meets_dreb_profile"]
    & (players["three_pct"] >= THRESHOLDS["big_3p_pct"])
    & (players["three_rate"] >= THRESHOLDS["big_3p_rate"])
    & players["meets_positive_ast_tov"]
)

players[["division", "name", "pos", "meets_all_player_baseline", "meets_pg_preferred", "meets_wing_preferred", "meets_big_preferred"]].head()

## Archetype Scores

Scores are 0-100. They mix percentile strength with the team thresholds. These weights are intentionally easy to tune after reviewing known players.

In [ ]:
def bool_bonus(flag, points):
    return np.where(flag, points, 0.0)

players["score_pg_combo"] = (
    0.35 * players["pct_assist_creation"]
    + 0.20 * players["pct_three_pct"]
    + 0.15 * players["pct_three_rate"]
    + 0.20 * players["pct_ast_tov"]
    + 0.10 * players["pct_efg"]
    + bool_bonus(players["meets_pg_preferred"], 8)
)

players["score_wing_2_4"] = (
    0.30 * players["pct_dreb_pos_adj"]
    + 0.25 * players["pct_three_pct"]
    + 0.20 * players["pct_three_rate"]
    + 0.15 * players["pct_ast_tov"]
    + 0.10 * players["pct_size"]
    + bool_bonus(players["meets_wing_preferred"], 8)
)

players["score_stretch_big"] = (
    0.30 * players["pct_dreb_pos_adj"]
    + 0.25 * players["pct_size"]
    + 0.20 * players["pct_three_pct"]
    + 0.15 * players["pct_three_rate"]
    + 0.10 * players["pct_ast_tov"]
    + bool_bonus(players["meets_big_preferred"], 8)
)

for col in ["score_pg_combo", "score_wing_2_4", "score_stretch_big"]:
    players[col] = players[col].clip(0, 100)

score_cols = list(ARCHETYPE_LABELS)
players["primary_score_col"] = players[score_cols].idxmax(axis=1)
players["primary_archetype"] = players["primary_score_col"].map(ARCHETYPE_LABELS)
players["primary_score"] = players[score_cols].max(axis=1)

preferred_flag_for_score = {
    "score_pg_combo": "meets_pg_preferred",
    "score_wing_2_4": "meets_wing_preferred",
    "score_stretch_big": "meets_big_preferred",
}

players["meets_primary_preferred"] = [
    bool(row[preferred_flag_for_score[row["primary_score_col"]]])
    for _, row in players.iterrows()
]

players[["division", "name", "pos", "primary_archetype", "primary_score", "score_pg_combo", "score_wing_2_4", "score_stretch_big"]].head()

## Fit Tiers And Notes

`Strong fit` means a high score plus the preferred threshold profile. `Partial fit` means a useful score but not every preferred box checked. `Outlier fit` captures players who miss the profile but have an elite trait.

In [ ]:
def fit_tier(row):
    if row["primary_score"] >= 75 and row["meets_primary_preferred"]:
        return "Strong fit"
    if row["primary_score"] >= 60:
        return "Partial fit"
    if max(row["pct_assist_creation"], row["pct_dreb_pos_adj"], row["pct_three_pct"]) >= 90 and row["primary_score"] >= 50:
        return "Outlier fit"
    return "Low fit"

def missing_notes(row):
    notes = []
    if not row["meets_high_assist"]:
        notes.append("assist creation below high threshold")
    if not row["meets_efg_500"]:
        notes.append("eFG below .500")
    if not row["meets_3p_300"]:
        notes.append("3P% below .300")
    if not row["meets_high_ast_tov"]:
        notes.append("A/TO below high threshold")
    if not row["meets_dreb_profile"]:
        notes.append("DReb profile not met")
    if not notes:
        return "meets all-player baseline"
    return "; ".join(notes)

players["fit_tier"] = players.apply(fit_tier, axis=1)
players["profile_notes"] = players.apply(missing_notes, axis=1)

players[["division", "name", "primary_archetype", "primary_score", "fit_tier", "profile_notes"]].head()

## Optional Archetype PCA Coordinates

These are new in-memory PCA coordinates based only on the archetype features above. They do not replace or modify any CSV columns.

In [ ]:
pca_features = [
    "pct_assist_creation",
    "pct_three_pct",
    "pct_three_rate",
    "pct_ast_tov",
    "pct_efg",
    "pct_dreb_pos_adj",
    "pct_size",
]

X = players[pca_features].fillna(players[pca_features].median()).to_numpy(dtype=float)
X = (X - X.mean(axis=0)) / np.where(X.std(axis=0) == 0, 1, X.std(axis=0))

try:
    from sklearn.decomposition import PCA
    coords = PCA(n_components=2, random_state=0).fit_transform(X)
except Exception:
    # Lightweight SVD fallback if sklearn is unavailable.
    _, _, vt = np.linalg.svd(X, full_matrices=False)
    coords = X @ vt[:2].T

players["arch_PC1_new"] = coords[:, 0]
players["arch_PC2_new"] = coords[:, 1]

players[["division", "name", "primary_archetype", "arch_PC1_new", "arch_PC2_new"]].head()

## Results Summary

In [ ]:
summary = (
    players.groupby(["division", "primary_archetype", "fit_tier"])
    .size()
    .rename("players")
    .reset_index()
    .sort_values(["division", "primary_archetype", "fit_tier"])
)

summary

## Top PG / Combo Guard Fits

In [ ]:
result_cols = [
    "division", "name", "team", "conference", "pos", "height_in", "mpg", "ppg", "apg", "rpg",
    "three_pct", "three_rate", "efg", "ast_tov", "assist_creation", "assist_source", "dreb", "dreb_source",
    "primary_archetype", "primary_score", "fit_tier", "score_pg_combo", "score_wing_2_4", "score_stretch_big",
    "meets_pg_preferred", "meets_all_player_baseline", "profile_notes",
]

top_pg_combo = (
    players.sort_values("score_pg_combo", ascending=False)
    .loc[:, result_cols]
    .head(50)
)

top_pg_combo

## Top Fits By Archetype

In [ ]:
top_by_archetype = (
    players.sort_values("primary_score", ascending=False)
    .groupby("primary_archetype", group_keys=False)
    .head(25)
    .loc[:, result_cols]
)

top_by_archetype

## Players Meeting The All-Player Baseline

In [ ]:
baseline_fits = (
    players[players["meets_all_player_baseline"]]
    .sort_values("primary_score", ascending=False)
    .loc[:, result_cols]
)

baseline_fits.head(100)

## Export Optional Results

This is disabled by default. Turning it on creates a separate results CSV and still does not modify the source datasets.

In [ ]:
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    export_cols = result_cols + [
        "pct_assist_creation", "pct_three_pct", "pct_three_rate", "pct_ast_tov", "pct_efg", "pct_dreb_pos_adj", "pct_size",
        "meets_wing_preferred", "meets_big_preferred", "arch_PC1_new", "arch_PC2_new",
    ]
    players.loc[:, export_cols].to_csv(HERE / "archetype_results.csv", index=False)
    print("Wrote archetype_results.csv")
else:
    print("EXPORT_RESULTS is False; no files written.")